# Create power flow summary statistics across timesteps

## Import packages and define functions

In [ ]:
## Time packages
import time
from datetime import datetime

## Memory packages
import psutil  # tracking memory and cpu usage
import resource  # tracking memory and cpu usage
import gc
import sys

## Data structure packages
import numpy as np
import pandas as pd  # to create data frames
import pyarrow as pa

## Math and logic packages
import random
import math
from decimal import Decimal

## Load & Save data packages
import os
import glob
import json
import yaml
import joblib

## Plotting packages
import matplotlib.pyplot as plt

## Type hints
from typing import List, Dict, Any, Iterable, Optional, Tuple, Union

## Internal functions packages
from src import figure_ops
from src import input_ops
from src import df_ops
from src import file_ops


## Define functions for Memory usage
## --- Memory functions ---
def print_memory_usage(msg=""):
    mem = psutil.virtual_memory()
    print(f"{msg} | Memory used: {mem.used / 1e9:.2f} GB / {mem.total / 1e9:.2f} GB")


def print_self_memory_usage(msg=""):
    usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # On Linux, ru_maxrss is in kilobytes
    print(f"{msg} | Memory used by this process: {usage / 1e6:.2f} GB")


def _bytes_human(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


def print_system_memory(msg=""):
    mem = psutil.virtual_memory()
    print(f"{msg} | System used: {_bytes_human(mem.used)} / {_bytes_human(mem.total)}\n")


def print_process_memory(msg=""):
    p = psutil.Process(os.getpid())
    m = p.memory_full_info()
    rss = getattr(m, "rss", None)
    peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss

    fields = [f"{msg} | Currently in RAM: {_bytes_human(rss)}"]
    fields.append(f"Peak RSS: {_bytes_human(int(peak_kb * 1024))}")

    print(" | ".join(fields))
    
# ---------------------------
# Percent-label helper functions
# ---------------------------
def _format_percent_value(x: Decimal) -> str:
    x = x.normalize()
    s = format(x, "f")

    if "." in s:
        s = s.rstrip("0").rstrip(".")

    return s


def parse_percent_label(percent_label: str) -> Tuple[Decimal, Decimal]:
    """
    Convert labels such as:
        'top_0_1%'        -> (Decimal("0"), Decimal("1"))
        'top_1-5%'        -> (Decimal("1"), Decimal("5"))
        'top_5-10%'       -> (Decimal("5"), Decimal("10"))
        'top_0-10%'       -> (Decimal("0"), Decimal("10"))
        'top_10-20%'      -> (Decimal("10"), Decimal("20"))
        'top_0_0.02%'     -> (Decimal("0"), Decimal("0.02"))
        'top_0.02_0.04%'  -> (Decimal("0.02"), Decimal("0.04"))
    """
    if not percent_label.startswith("top_") or not percent_label.endswith("%"):
        raise ValueError(
            f"Invalid percent label: {percent_label}. "
            "Expected format like 'top_0_1%', 'top_1-5%', or 'top_0.02_0.04%'."
        )

    body = percent_label.replace("top_", "").replace("%", "")

    if "-" in body:
        parts = body.split("-")
    elif "_" in body:
        parts = body.split("_")
    else:
        raise ValueError(
            f"Invalid percent label: {percent_label}. "
            "Expected a range, e.g., 'top_0_1%', 'top_1-5%', or 'top_0.02_0.04%'."
        )

    if len(parts) != 2:
        raise ValueError(
            f"Invalid percent label: {percent_label}. "
            "Expected exactly two values defining a range."
        )

    start_pct, end_pct = parts

    return Decimal(start_pct), Decimal(end_pct)


def get_merged_percent_file_suffix(percent_to_analyze: List[str]) -> str:
    """
    Convert:
        ['top_0_1%', 'top_1-5%', 'top_5-10%'] -> 'top_0_10_percent'
        ['top_0-10%', 'top_10-20%']           -> 'top_0_20_percent'
        ['top_0_0.02%', 'top_0.02_0.04%']     -> 'top_0_0.04_percent'
    """
    if len(percent_to_analyze) == 0:
        raise ValueError("`percent_to_analyze` must contain at least one percent range.")

    parsed_ranges = [parse_percent_label(label) for label in percent_to_analyze]

    start_pct = min(start for start, end in parsed_ranges)
    end_pct = max(end for start, end in parsed_ranges)

    start_str = _format_percent_value(start_pct)
    end_str = _format_percent_value(end_pct)

    return f"top_{start_str}_{end_str}_percent"


def validate_percent_ranges_are_contiguous(percent_to_analyze: List[str]) -> None:
    parsed_ranges = sorted([parse_percent_label(label) for label in percent_to_analyze])

    for i in range(1, len(parsed_ranges)):
        prev_start, prev_end = parsed_ranges[i - 1]
        cur_start, cur_end = parsed_ranges[i]

        if cur_start != prev_end:
            raise ValueError(
                "Percent ranges are not contiguous. "
                f"Previous range: {_format_percent_value(prev_start)}-{_format_percent_value(prev_end)}%, "
                f"current range: {_format_percent_value(cur_start)}-{_format_percent_value(cur_end)}%."
            )

            
# ---------------------------
# Helper functions
# ---------------------------
def validate_input_file_exists(path: str, missing_file_behavior: str = "raise") -> bool:
    if os.path.exists(path):
        return True

    msg = f"Missing input file: {path}"

    if missing_file_behavior == "raise":
        raise FileNotFoundError(msg)

    if missing_file_behavior == "skip":
        print(msg)
        print("Skipping this file.\n")
        return False

    raise ValueError("`missing_file_behavior` must be either 'raise' or 'skip'.")


def get_predictions_dir(
    output_pf_path: str,
    city: str,
    region: str,
    TGW_scenario: str,
    TGW_weather_year: str,
    solar_battery_scenario_folder: str,
) -> str:
    return os.path.join(
        output_pf_path,
        f"{city}/{region}/{TGW_scenario}/{TGW_weather_year}/{solar_battery_scenario_folder}",
    )


def get_asset_file_path(
    predictions_dir: str,
    asset_name: str,
    merged_percent_file_suffix: str,
) -> str:
    return os.path.join(
        predictions_dir,
        f"{asset_name}_{merged_percent_file_suffix}.joblib",
    )


def get_asset_summary_save_path(
    predictions_dir_future: str,
    asset_name: str,
    merged_percent_file_suffix: str,
) -> str:
    return os.path.join(
        predictions_dir_future,
        f"{asset_name}_{merged_percent_file_suffix}_summary.joblib",
    )


def extract_df(
    nested_dict: dict,
    weather_year,
    scenario,
    smart_ds_year,
    city,
    region,
) -> pd.DataFrame:
    """
    Extract a DataFrame from nested joblib dicts.
    """
    weather_year_candidates = [weather_year, str(weather_year)]
    smart_ds_year_candidates = [smart_ds_year, str(smart_ds_year)]

    outer_keys_to_try = [
        (wy, scenario)
        for wy in weather_year_candidates
    ]

    inner_keys_to_try = [
        (sdy, city, region)
        for sdy in smart_ds_year_candidates
    ]

    for outer_key in outer_keys_to_try:
        if outer_key not in nested_dict:
            continue

        for inner_key in inner_keys_to_try:
            if inner_key in nested_dict[outer_key]:
                return nested_dict[outer_key][inner_key].copy()

    raise KeyError(
        "Could not find DataFrame in nested dictionary. "
        f"Tried outer keys: {outer_keys_to_try}. "
        f"Tried inner keys: {inner_keys_to_try}."
    )


def validate_required_cols(
    df: pd.DataFrame,
    required_cols: set,
    label: str,
) -> None:
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        print(f"Warning: {label} missing columns: {missing_cols}")


def make_static_df(
    df_baseline: pd.DataFrame,
    df_future: pd.DataFrame,
    asset_id_col: str,
    static_cols: List[str],
) -> pd.DataFrame:
    available_static_cols = [
        col for col in static_cols
        if col in df_baseline.columns and col in df_future.columns
    ]

    if asset_id_col not in available_static_cols:
        raise ValueError(f"{asset_id_col} must be included in static columns.")

    static_df = (
        pd.concat(
            [
                df_baseline[available_static_cols],
                df_future[available_static_cols],
            ],
            ignore_index=True,
        )
        .drop_duplicates(subset=[asset_id_col])
        .set_index(asset_id_col)
    )

    return static_df


def scenario_summary_asset(
    df: pd.DataFrame,
    asset_id_col: str,
    scenario_tag: str,
    thresholds: List[int],
) -> pd.DataFrame:
    """
    Compute per-asset summary stats on 'Loading [%]' across all selected rows in the given df.
    """
    grp = df.groupby(asset_id_col)["Loading [%]"]

    out = pd.DataFrame({
        f"90th_loading_{scenario_tag}": grp.quantile(0.90),
        f"95th_loading_{scenario_tag}": grp.quantile(0.95),
        f"99th_loading_{scenario_tag}": grp.quantile(0.99),
        f"min_loading_{scenario_tag}": grp.min(),
        f"max_loading_{scenario_tag}": grp.max(),
        f"median_loading_{scenario_tag}": grp.median(),
        f"average_loading_{scenario_tag}": grp.mean(),
        f"n_hours_{scenario_tag}": grp.size(),
    })

    out[f"n_unique_hours_{scenario_tag}"] = (
        df[[asset_id_col, "row_i"]]
        .drop_duplicates()
        .groupby(asset_id_col)["row_i"]
        .nunique()
    )

    for thr in thresholds:
        tmp = (
            df.loc[df["Loading [%]"] >= thr, [asset_id_col, "row_i"]]
            .drop_duplicates()
        )

        hours_over_col = f"hours_over_{thr}_loading_{scenario_tag}"

        if len(tmp) == 0:
            out[hours_over_col] = 0
        else:
            out[hours_over_col] = (
                tmp.groupby(asset_id_col)["row_i"]
                .nunique()
            )

        out[hours_over_col] = (
            out[hours_over_col]
            .fillna(0)
            .astype(int)
        )

    hours_over_100_col = f"hours_over_100_loading_{scenario_tag}"
    share_over_100_col = f"share_hours_over_100_loading_{scenario_tag}"
    n_unique_hours_col = f"n_unique_hours_{scenario_tag}"

    if hours_over_100_col in out.columns:
        out[share_over_100_col] = (
            out[hours_over_100_col] / out[n_unique_hours_col]
        ).replace([np.inf, -np.inf], np.nan)

    return out


def difference_stats_asset(
    df_baseline: pd.DataFrame,
    df_future: pd.DataFrame,
    asset_id_col: str,
    baseline_tag: str,
    future_tag: str,
    static_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compute future - baseline loading difference using the same MDH matching logic as the original code.

    Matching keys:
        asset_id_col, month, day, hour
    """
    cols_mdh = [asset_id_col, "month", "day", "hour", "Loading [%]"]

    b_mdh = df_baseline[cols_mdh].rename(columns={"Loading [%]": "loading_baseline"})
    f_mdh = df_future[cols_mdh].rename(columns={"Loading [%]": "loading_future"})

    merged_mdh = pd.merge(
        b_mdh,
        f_mdh,
        on=[asset_id_col, "month", "day", "hour"],
        how="inner",
    )

    diff_suf = f"{future_tag}_vs_{baseline_tag}"

    if len(merged_mdh) == 0:
        diff_stats = pd.DataFrame(
            index=static_df.index,
            data={
                f"min_loading_diff_{diff_suf}": np.nan,
                f"average_loading_diff_{diff_suf}": np.nan,
                f"max_loading_diff_{diff_suf}": np.nan,
            },
        )
    else:
        merged_mdh = merged_mdh.assign(
            diff=merged_mdh["loading_future"] - merged_mdh["loading_baseline"]
        )

        gdiff = merged_mdh.groupby(asset_id_col)["diff"]

        diff_stats = pd.DataFrame({
            f"min_loading_diff_{diff_suf}": gdiff.min(),
            f"average_loading_diff_{diff_suf}": gdiff.mean(),
            f"max_loading_diff_{diff_suf}": gdiff.max(),
        })

    del b_mdh, f_mdh, merged_mdh
    gc.collect()

    return diff_stats


def order_cols_asset(
    df: pd.DataFrame,
    static_cols: List[str],
    baseline_tag: str,
    future_tag: str,
    thresholds: List[int],
) -> List[str]:
    diff_suf = f"{future_tag}_vs_{baseline_tag}"

    stat_names = [
        "90th_loading",
        "95th_loading",
        "99th_loading",
        "min_loading",
        "max_loading",
        "median_loading",
        "average_loading",
        "n_hours",
        "n_unique_hours",
    ]

    for thr in thresholds:
        stat_names.append(f"hours_over_{thr}_loading")

    stat_names.append("share_hours_over_100_loading")

    baseline_cols = [f"{s}_{baseline_tag}" for s in stat_names]
    future_cols = [f"{s}_{future_tag}" for s in stat_names]

    diff_cols = [
        f"min_loading_diff_{diff_suf}",
        f"average_loading_diff_{diff_suf}",
        f"max_loading_diff_{diff_suf}",
    ]

    ordered = (
        [c for c in static_cols if c in df.columns]
        + [c for c in baseline_cols if c in df.columns]
        + [c for c in future_cols if c in df.columns]
        + [c for c in diff_cols if c in df.columns]
    )

    leftovers = [c for c in df.columns if c not in ordered]

    return ordered + leftovers


def summarize_asset_pair(
    asset_name: str,
    asset_config: dict,
    smart_ds_year,
    city: str,
    region: str,
    baseline_TGW_year: str,
    baseline_TGW_scenario: str,
    future_TGW_year: str,
    future_TGW_scenario: str,
    output_pf_path: str,
    solar_battery_scenario_folder: str,
    merged_percent_file_suffix: str,
    thresholds: List[int],
    missing_file_behavior: str = "raise",
    display_outputs: bool = True,
) -> Optional[str]:

    asset_id_col = asset_config["asset_id_col"]
    static_cols = asset_config["static_cols"]
    required_cols = asset_config["required_cols"]

    baseline_tag = f"{baseline_TGW_scenario}_{baseline_TGW_year}"
    future_tag = f"{future_TGW_scenario}_{future_TGW_year}"

    predictions_dir_baseline = get_predictions_dir(
        output_pf_path=output_pf_path,
        city=city,
        region=region,
        TGW_scenario=baseline_TGW_scenario,
        TGW_weather_year=baseline_TGW_year,
        solar_battery_scenario_folder=solar_battery_scenario_folder,
    )

    predictions_dir_future = get_predictions_dir(
        output_pf_path=output_pf_path,
        city=city,
        region=region,
        TGW_scenario=future_TGW_scenario,
        TGW_weather_year=future_TGW_year,
        solar_battery_scenario_folder=solar_battery_scenario_folder,
    )

    path_baseline = get_asset_file_path(
        predictions_dir=predictions_dir_baseline,
        asset_name=asset_name,
        merged_percent_file_suffix=merged_percent_file_suffix,
    )

    path_future = get_asset_file_path(
        predictions_dir=predictions_dir_future,
        asset_name=asset_name,
        merged_percent_file_suffix=merged_percent_file_suffix,
    )

    save_path = get_asset_summary_save_path(
        predictions_dir_future=predictions_dir_future,
        asset_name=asset_name,
        merged_percent_file_suffix=merged_percent_file_suffix,
    )

    print(f"--- {asset_name} ---")
    print(f"Baseline path: {path_baseline}")
    print(f"Future path:   {path_future}")
    print(f"Save path:     {save_path}\n")

    if not validate_input_file_exists(path_baseline, missing_file_behavior):
        return None

    if not validate_input_file_exists(path_future, missing_file_behavior):
        return None

    print_process_memory(f"Before loading {asset_name}")

    baseline_obj = joblib.load(path_baseline)
    future_obj = joblib.load(path_future)

    df_baseline = extract_df(
        nested_dict=baseline_obj,
        weather_year=baseline_TGW_year,
        scenario=baseline_TGW_scenario,
        smart_ds_year=smart_ds_year,
        city=city,
        region=region,
    )

    df_future = extract_df(
        nested_dict=future_obj,
        weather_year=future_TGW_year,
        scenario=future_TGW_scenario,
        smart_ds_year=smart_ds_year,
        city=city,
        region=region,
    )

    del baseline_obj, future_obj
    gc.collect()

    validate_required_cols(
        df=df_baseline,
        required_cols=required_cols,
        label=f"baseline ({asset_name})",
    )

    validate_required_cols(
        df=df_future,
        required_cols=required_cols,
        label=f"future ({asset_name})",
    )

    df_baseline = df_baseline.copy()
    df_future = df_future.copy()

    df_baseline["__scenario__"] = baseline_tag
    df_future["__scenario__"] = future_tag

    static_df = make_static_df(
        df_baseline=df_baseline,
        df_future=df_future,
        asset_id_col=asset_id_col,
        static_cols=static_cols,
    )

    summ_baseline = scenario_summary_asset(
        df=df_baseline,
        asset_id_col=asset_id_col,
        scenario_tag=baseline_tag,
        thresholds=thresholds,
    )

    summ_future = scenario_summary_asset(
        df=df_future,
        asset_id_col=asset_id_col,
        scenario_tag=future_tag,
        thresholds=thresholds,
    )

    diff_stats = difference_stats_asset(
        df_baseline=df_baseline,
        df_future=df_future,
        asset_id_col=asset_id_col,
        baseline_tag=baseline_tag,
        future_tag=future_tag,
        static_df=static_df,
    )

    summary_df = (
        static_df
        .join(summ_baseline, how="outer")
        .join(summ_future, how="outer")
        .join(diff_stats, how="left")
        .reset_index()
    )

    summary_df = summary_df[
        order_cols_asset(
            df=summary_df,
            static_cols=static_cols,
            baseline_tag=baseline_tag,
            future_tag=future_tag,
            thresholds=thresholds,
        )
    ]

    summary_dict_by_scenario = {}
    summary_dict_by_scenario[(future_TGW_year, future_TGW_scenario)] = {}
    summary_dict_by_scenario[(future_TGW_year, future_TGW_scenario)][
        (smart_ds_year, city, region)
    ] = summary_df

    joblib.dump(summary_dict_by_scenario, save_path)

    print(f"{asset_name.capitalize()} summarized: {len(summary_df)}")
    print(f"Saved: {save_path}\n")

    if display_outputs:
        display(summary_df.head(3))

    del df_baseline
    del df_future
    del static_df
    del summ_baseline
    del summ_future
    del diff_stats
    del summary_df
    del summary_dict_by_scenario
    gc.collect()

    print_process_memory(f"After {asset_name} memory deletion")
    print()

    return save_path

## Load config file with scenarios and parameters 

In [ ]:
# --- Load config ---
config_file_name = "opendss_config1"
config_path = f"config/{config_file_name}.yaml"
config = input_ops.load_config(config_path)

CITY_REGIONS_TO_RUN = config["CITY_REGIONS_TO_RUN"]

smart_ds_years = config["smart_ds_years"]

## Initialize parameters for saving paths
output_pf_path = config["output_pf_path"]

solution_mode = config.get("solution_mode", None)

demand_mode = config.get("demand_mode", None)

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)


start_row_percent = config["start_row_percent"]
top_percent_mdh = config["top_percent_mdh"]

percent_to_analyze = [
    f"top_{_format_percent_value(Decimal(str(start_row_percent)))}-{_format_percent_value(Decimal(str(top_percent_mdh)))}%"
]

print(
    f"start_row_percent: {start_row_percent}\n"
    f"top_percent_mdh: {top_percent_mdh}\n"
    f"percent_to_analyze: {percent_to_analyze}"
)


TGW_years_scenarios_ranges = config.get("TGW_years_scenarios_ranges", [])

comparison_TGW_scenario = "rcp45hotter"

# Historical and future weather-year ranges to compare
historical_range = next(
    r for r in TGW_years_scenarios_ranges
    if "historical" in r["scenarios"]
)

future_range = next(
    r for r in TGW_years_scenarios_ranges
    if comparison_TGW_scenario in r["scenarios"]
)

historical_years = range(
    int(historical_range["start_year"]),
    int(historical_range["end_year"]) + 1,
)

future_years = range(
    int(future_range["start_year"]),
    int(future_range["end_year"]) + 1,
)

if len(historical_years) != len(future_years):
    raise ValueError(
        "Historical and future periods must contain the same number of years."
    )

thresholds = [70, 80, 90, 100]

missing_file_behavior = "raise"  # options: "raise", "skip"

display_outputs = True

assets_to_summarize = {
    "transformers": {
        "asset_id_col": "Transformer",
        "static_cols": [
            "Transformer",
            "Bus",
            "Lat",
            "Long",
            "Num windings",
            "Num phases",
            "KV rating [kV]",
            "kVA rating [kVA]",
        ],
        "required_cols": {
            "Transformer",
            "Loading [%]",
            "row_i",
            "month",
            "day",
            "hour",
            "Bus",
            "Lat",
            "Long",
            "Num windings",
            "Num phases",
            "KV rating [kV]",
            "kVA rating [kVA]",
        },
    },
    "lines": {
        "asset_id_col": "Line",
        "static_cols": [
            "Line",
            "LineCode",
            "Length [km]",
            "From_Lat",
            "To_Lat",
            "From_Long",
            "To_Long",
            "NormAmps [A]",
            "Nominal V [kV]",
        ],
        "required_cols": {
            "month",
            "day",
            "hour",
            "row_i",
            "Line",
            "LineCode",
            "Length [km]",
            "From_Lat",
            "To_Lat",
            "From_Long",
            "To_Long",
            "NormAmps [A]",
            "Nominal V [kV]",
            "Loading [%]",
        },
    },
}


validate_percent_ranges_are_contiguous(percent_to_analyze)
merged_percent_file_suffix = get_merged_percent_file_suffix(percent_to_analyze)

## Create dictionary w/ summary statistics

In [ ]:
start_time = time.time()

saved_summary_paths = []

for smart_ds_year in smart_ds_years:

    print(f"\n==================== smart_ds_year: {smart_ds_year} ====================\n")

    
    for baseline_TGW_year, future_TGW_year in zip(
        historical_years,
        future_years,
    ):

        baseline_TGW_year = str(baseline_TGW_year)
        future_TGW_year = str(future_TGW_year)

        baseline_TGW_scenario = "historical"
        future_TGW_scenario = comparison_TGW_scenario    

        print(
            f"\n==================== Comparison: "
            f"{baseline_TGW_scenario}_{baseline_TGW_year} vs "
            f"{future_TGW_scenario}_{future_TGW_year} ====================\n"
        )

        for city, regions in CITY_REGIONS_TO_RUN.items():
            for region in regions:

                print(f"\n-------------------- city: {city} region: {region} --------------------\n")

                for asset_name, asset_config in assets_to_summarize.items():

                    save_path = summarize_asset_pair(
                        asset_name=asset_name,
                        asset_config=asset_config,
                        smart_ds_year=smart_ds_year,
                        city=city,
                        region=region,
                        baseline_TGW_year=baseline_TGW_year,
                        baseline_TGW_scenario=baseline_TGW_scenario,
                        future_TGW_year=future_TGW_year,
                        future_TGW_scenario=future_TGW_scenario,
                        output_pf_path=output_pf_path,
                        solar_battery_scenario_folder=solar_battery_scenario_folder,
                        merged_percent_file_suffix=merged_percent_file_suffix,
                        thresholds=thresholds,
                        missing_file_behavior=missing_file_behavior,
                        display_outputs=display_outputs,
                    )

                    if save_path is not None:
                        saved_summary_paths.append(save_path)

end_time = time.time()
print("Runtime:", (end_time - start_time) / 60, "minutes")
print(f"Number of saved summary files: {len(saved_summary_paths)}")